# ModernFloraBERT: ModernBERT as the base model

This notebook runs the full pipeline with **ModernBERT** (instead of RoBERTa) as the
base transformer, for maize promoter gene-expression prediction:

1. Train a byte-level BPE tokenizer with ModernBERT special tokens (`[CLS]/[SEP]/[PAD]/[UNK]/[MASK]`)
2. Pretrain a masked-LM ModernBERT (small 6-layer config from `config.yaml`)
3. Finetune the mean-pooling 8-tissue regression head
4. Evaluate

## How code & data are obtained
- **Code** is cloned from this repo: `https://github.com/darshleenkaur01/ModernFloraBERT.git`
  (make sure the ModernBERT changes are committed and pushed).
- **Data** (and the RoBERTa baseline tokenizer) is copied from the attached
  **`gurveersinghvirk/florabert-base`** dataset — no need to re-upload it.
- Data expected inside the dataset:
  - Pretrain: `data/final/transformer/seq/all_seqs_train.txt` + `all_seqs_test.txt`
  - Finetune: `data/final/transformer/genex/nam/{train,eval,test}.tsv`
- **GPU accelerator** (this notebook is GPU-only, no TPU).

Run cells top-to-bottom.

In [ ]:
# Clean slate
!rm -rf /kaggle/working/florabert
print("cleaned /kaggle/working/florabert")

In [ ]:
# Code comes from this repo (git clone); data comes from the attached dataset.
import os, shutil, subprocess

REPO_URL = "https://github.com/darshleenkaur01/ModernFloraBERT.git"
WORK = "/kaggle/working/florabert"
DATASET = "/kaggle/input/florabert-base/florabert"

# 1) Clone the code (shallow)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK], check=True)

# 2) Copy data (and the RoBERTa baseline tokenizer) from the dataset
os.chdir(WORK)
shutil.copytree(os.path.join(DATASET, "data"), os.path.join(WORK, "data"), dirs_exist_ok=True)
tok_src = os.path.join(DATASET, "models", "byte-level-bpe-tokenizer")
if os.path.exists(tok_src):
    shutil.copytree(tok_src, os.path.join(WORK, "models", "byte-level-bpe-tokenizer"), dirs_exist_ok=True)
print("Code cloned from", REPO_URL)
print("Data copied from", DATASET)

In [ ]:
# GPU install: keep torch from the Kaggle image, upgrade transformers to a
# ModernBERT-compatible 4.x release (>=4.48, <5.0) plus the project deps.
!pip install -q --upgrade \
    "transformers>=4.48,<5.0" \
    "tokenizers>=0.20" \
    "datasets>=2.14,<3.0" \
    "accelerate>=0.30" \
    safetensors torch_optimizer scipy biopython wget
print("installed")

In [ ]:
import os, sys
os.chdir('/kaggle/working/florabert')
sys.path.insert(0, '/kaggle/working/florabert')

from module.florabert import config

assert "modernbert-base" in config.settings["models"], (
    "config.yaml has no 'modernbert-base' entry. The cloned repo is stale - "
    "commit & push the ModernBERT changes to "
    "https://github.com/darshleenkaur01/ModernFloraBERT.git and re-run."
)
print("Model entries:", list(config.settings["models"]))

seq_dir = config.data_final / "transformer" / "seq"
for f in ["all_seqs_train.txt", "all_seqs_test.txt"]:
    print("pretrain data", f, "->", (seq_dir / f).exists())

genex_dir = config.data_final / "transformer" / "genex" / "nam"
for f in ["train.tsv", "eval.tsv", "test.tsv"]:
    print("finetune data", f, "->", (genex_dir / f).exists())

## Step 1 — Train the byte-level BPE tokenizer (ModernBERT)

Trains with `[CLS]/[SEP]/[PAD]/[UNK]/[MASK]` (ids 0..4) and saves a fast-tokenizer
directory `models/modernbert-byte-level-bpe-tokenizer/` containing `tokenizer.json`
(ModernBERT is loaded via `PreTrainedTokenizerFast`, which needs that file).

In [ ]:
!python scripts/0-data-loading-processing/07_train_tokenizer.py --model modernbert

In [ ]:
import gc
import torch
from module.florabert import config, utils, transformers as tr

settings = utils.get_model_settings(config.settings, model_name="modernbert-lm")
config_obj, tokenizer, model = tr.load_model(
    "modernbert-lm",
    config.tokenizer_dir_for_model("modernbert-lm"),
    **settings,
)
print("Tokenizer vocab size:", len(tokenizer))
print("Special tokens:", {k: getattr(tokenizer, k, None) for k in
      ["cls_token", "sep_token", "pad_token", "unk_token", "mask_token"]})
print("ModernBERT LM params:", sum(p.numel() for p in model.parameters()))

# quick forward smoke test on CPU
inputs = tokenizer("ACGTACGTACGTTTTAAACCCGGG", return_tensors="pt")
model.eval()
with torch.no_grad():
    out = model(**inputs)
print("MLM logits shape:", tuple(out.logits.shape))

del model, tokenizer, config_obj
gc.collect()

## Step 2 — Pretrain the language model (masked LM)

Trains `ModernBertForMaskedLM` from scratch on plant promoter sequences with the
`lamb` optimizer + linear schedule. Checkpoints are saved to
`models/transformer/language-model-modernbert/`.

In [ ]:
!python scripts/1-modeling/pretrain.py --model-name modernbert-lm

## Step 3 — Finetune the multitask regression head

Loads `models/transformer/language-model-modernbert/`, adds the project's
mean-pooling regression head (8 tissues) and trains with `lamb` + constant
schedule. Saved to `models/transformer/prediction-model-modernbert/`.

In [ ]:
!python scripts/1-modeling/finetune.py --model-name modernbert-pred-mean-pool

## Step 4 — Evaluate

Computes per-tissue MSE/MAE/R2 (and per-NAM-line) on the test split and writes
`output/model_eval/florabert.csv` + scatter plots to `output/transformer/`.

In [ ]:
!python scripts/1-modeling/evaluate.py --model-name modernbert-pred-mean-pool

## Optional — RoBERTa baseline for the head-to-head comparison

The original RoBERTa tokenizer (`models/byte-level-bpe-tokenizer`) ships in the
dataset. Run the exact same three stages with the RoBERTa model names to get a
controlled comparison (same data/splits, same 6-layer config, same hyperparameters).
Checkpoints land in `models/transformer/language-model` and
`models/transformer/prediction-model`.

In [ ]:
# !python scripts/1-modeling/pretrain.py --model-name roberta-lm
# !python scripts/1-modeling/finetune.py --model-name roberta-pred-mean-pool
# !python scripts/1-modeling/evaluate.py --model-name roberta-pred-mean-pool

## Outputs
- Tokenizer: `models/modernbert-byte-level-bpe-tokenizer/`
- Pretrained LM: `models/transformer/language-model-modernbert/`
- Finetuned model: `models/transformer/prediction-model-modernbert/`
- Evaluation: `output/model_eval/florabert.csv`

For a clean RoBERTa-vs-ModernBERT comparison, both models use identical data,
6-layer/6-head/768 config (ModernBERT `intermediate_size=2048` keeps the GeGLU MLP
param-matched to RoBERTa's 3072-wide MLP), `lamb` optimizer, and the same
schedulers/seeds.